# LAB02BIA - Tarefa 05

## Previsao de rotatividade de clientes (Churn)

Este notebook foi organizado e comentado a partir do arquivo original. O fluxo e:

1. Importar e inspecionar a base de dados;
2. Tratar valores ausentes;
3. Renomear e preparar as variaveis;
4. Realizar uma Analise Exploratoria de Dados (EDA);
5. Estimar a importancia das caracteristicas com **Random Forest**;
6. Treinar uma **Regressao Logistica** para classificar a rotatividade.

> **Importante:** ajuste o caminho de `ARQUIVO_DADOS` para o local onde esta o arquivo `Rotatividade_clientes.csv` no seu computador. No VSCode, e recomendavel colocar o CSV na mesma pasta deste notebook.

## 1. Bibliotecas e configuracoes

As bibliotecas abaixo permitem manipular tabelas (`pandas`), realizar calculos numericos (`numpy`), criar graficos (`matplotlib` e `seaborn`) e construir os modelos de Machine Learning (`scikit-learn`).

In [ ]:
# Oculta avisos que nao interrompem a execucao do notebook.
import warnings
warnings.filterwarnings('ignore')

# Manipulacao de dados tabulares.
import pandas as pd

# Operacoes numericas e ordenacao de vetores.
import numpy as np

# Bibliotecas para visualizacao de dados.
import matplotlib.pyplot as plt
import seaborn as sns

# Configuracao visual padrao para os graficos.
sns.set_theme(style='whitegrid')

## 2. Leitura e inspecao inicial da base

A instrucao `pd.read_csv()` carrega o arquivo CSV como um DataFrame, isto e, uma tabela do `pandas`. As proximas celulas verificam quantidade de linhas, dimensoes, tipos de variaveis e valores ausentes.

In [ ]:
# Caminho do arquivo de dados.
# Se o CSV estiver na mesma pasta do notebook, deixe somente o nome do arquivo.
ARQUIVO_DADOS = 'Rotatividade_clientes.csv'

# Le a base de dados e armazena a tabela na variavel "dados".
dados = pd.read_csv(ARQUIVO_DADOS)

# Exibe as cinco primeiras linhas para conferir se a leitura foi realizada corretamente.
dados.head(5)

In [ ]:
# Retorna o numero total de registros (linhas) da base.
len(dados)

In [ ]:
# Retorna (numero_de_linhas, numero_de_colunas).
dados.shape

In [ ]:
# Verifica se existe pelo menos um valor ausente em toda a tabela.
dados.isnull().values.any()

In [ ]:
# Indica, para cada coluna, se ha algum valor ausente.
dados.isnull().any()

In [ ]:
# Estatisticas descritivas de salario estimado e idade.
# Para colunas numericas, o describe mostra contagem, media, desvio-padrao, minimo, quartis e maximo.
dados[['EstimatedSalary', 'Age']].describe()

In [ ]:
# Estatisticas descritivas de todas as colunas numericas da base.
dados.describe()

In [ ]:
# Conta quantos clientes possuem ou nao possuem cartao de credito.
dados['HasCrCard'].value_counts()

## 3. Tratamento de valores ausentes

O preenchimento adotado no arquivo original e:
- `EstimatedSalary`: media;
- `Gender`: categoria mais frequente (moda);
- `Age`: valor mais frequente (moda).

Essas escolhas devem ser justificadas conforme o contexto da base. Para uma aplicacao real, o ideal e calcular os valores de imputacao apenas no conjunto de treino, evitando vazamento de dados. Aqui o objetivo e reproduzir e explicar a atividade.

In [ ]:
# Conta a quantidade de valores ausentes em cada coluna.
dados.isnull().sum()

In [ ]:
# Calcula o percentual de valores ausentes por coluna.
percentual_ausentes = round(dados.isnull().sum() / len(dados) * 100, 2)
percentual_ausentes

In [ ]:
# Calcula a media dos salarios estimados validos.
valor_medio = dados['EstimatedSalary'].mean()

# Substitui valores ausentes de salario pela media calculada.
dados['EstimatedSalary'] = dados['EstimatedSalary'].fillna(valor_medio)

In [ ]:
# Conta as ocorrencias de cada categoria de genero.
dados['Gender'].value_counts()

In [ ]:
# idxmax() retorna a categoria mais frequente.
# Essa categoria sera usada para preencher os valores ausentes de genero.
genero_mais_frequente = dados['Gender'].value_counts().idxmax()
dados['Gender'] = dados['Gender'].fillna(genero_mais_frequente)

In [ ]:
# mode() retorna a(s) moda(s), isto e, o(s) valor(es) mais frequente(s).
valor_moda = dados['Age'].mode()

# [0] seleciona a primeira moda; em seguida, preenche as idades ausentes.
dados['Age'] = dados['Age'].fillna(valor_moda[0])

In [ ]:
# Confirmacao: apos a imputacao, o esperado e obter False.
dados.isnull().values.any()

## 4. Preparacao das variaveis

Nesta etapa, nomes de colunas sao traduzidos, a identificacao unica do cliente e removida e a variavel-alvo e posicionada ao final da tabela. Em seguida, as variaveis categoricas sao definidas com o tipo `category`.

In [ ]:
# Renomeia colunas para nomes mais claros em portugues.
dados = dados.rename(columns={
    'CredRate': 'EscoreCredito',
    'ActMem': 'MembroAtivo',
    'Prod Number': 'NumDeProdutos',
    'Exited': 'Rotatividade'
})

# Mostra os nomes atuais das colunas.
dados.columns

In [ ]:
# CustomerId e um identificador unico e nao representa uma caracteristica causal do comportamento.
# Portanto, essa coluna e removida antes da modelagem.
dados.drop(labels=['CustomerId'], axis=1, inplace=True)

In [ ]:
# Guarda temporariamente a variavel-alvo.
coluna_rotatividade = dados['Rotatividade']

# Remove a coluna e a reinsere na ultima posicao da tabela.
dados.drop(labels=['Rotatividade'], axis=1, inplace=True)
dados.insert(len(dados.columns), 'Rotatividade', coluna_rotatividade.values)

# Confirma a ordem atual das colunas.
dados.columns

In [ ]:
# Converte as colunas qualitativas para o tipo categorico do pandas.
for coluna in ['Geography', 'Gender', 'HasCrCard', 'Rotatividade', 'MembroAtivo']:
    dados[coluna] = dados[coluna].astype('category')

# Verifica o tipo de dados de cada variavel.
dados.dtypes

## 5. Analise Exploratoria de Dados (EDA)

Os graficos a seguir permitem observar distribuicoes, possiveis diferencas entre clientes que sairam e os que permaneceram, e relacoes preliminares entre as caracteristicas e a rotatividade.

In [ ]:
# Quantidade absoluta de clientes sem rotatividade (0) e com rotatividade (1).
dados['Rotatividade'].value_counts()

In [ ]:
# Percentual de clientes em cada classe de rotatividade.
round(dados['Rotatividade'].value_counts(normalize=True) * 100, 2)

In [ ]:
# Percentual de clientes que possuem (1) ou nao possuem (0) cartao de credito.
round(dados['HasCrCard'].value_counts(normalize=True) * 100, 2)

In [ ]:
# Resume novamente as variaveis numericas apos o tratamento da base.
dados.describe()

In [ ]:
# Cria tres paineis para comparar as distribuicoes de salario, idade e saldo.
fig, axes = plt.subplots(ncols=3, figsize=(15, 6))

sns.histplot(dados['EstimatedSalary'], kde=True, color='gray', ax=axes[0])
axes[0].set_title('Salario estimado')
axes[0].set_ylabel('Numero de consumidores')

sns.histplot(dados['Age'], kde=True, color='gray', ax=axes[1])
axes[1].set_title('Idade')
axes[1].set_ylabel('Numero de consumidores')

sns.histplot(dados['Balance'], kde=True, color='gray', ax=axes[2])
axes[2].set_title('Saldo')
axes[2].set_ylabel('Numero de consumidores')

plt.tight_layout()
plt.show()

In [ ]:
# Compara a distribuicao de Gender entre clientes com e sem rotatividade.
plt.figure(figsize=(15, 4))
ax = sns.countplot(y='Gender', hue='Rotatividade', data=dados, palette='Greys_r')
ax.set_title('Distribuicao da rotatividade dos consumidores por genero')
ax.legend(title='Rotatividade', labels=['Nenhuma rotatividade', 'Rotatividade'])
plt.tight_layout()
plt.show()

In [ ]:
# Compara a distribuicao de Geography entre clientes com e sem rotatividade.
plt.figure(figsize=(15, 4))
ax = sns.countplot(x='Geography', hue='Rotatividade', data=dados, palette='Greys_r')
ax.set_title('Distribuicao geografica dos consumidores')
ax.legend(title='Rotatividade', labels=['Nenhuma rotatividade', 'Rotatividade'])
plt.tight_layout()
plt.show()

In [ ]:
# Compara a distribuicao de NumDeProdutos entre clientes com e sem rotatividade.
plt.figure(figsize=(15, 4))
ax = sns.countplot(x='NumDeProdutos', hue='Rotatividade', data=dados, palette='Greys_r')
ax.set_title('Distribuicao dos consumidores por numero de produtos')
ax.legend(title='Rotatividade', labels=['Nenhuma rotatividade', 'Rotatividade'])
plt.tight_layout()
plt.show()

In [ ]:
# Compara a distribuicao de idade do consumidor para as duas classes da variavel-alvo.
plt.figure(figsize=(15, 4))
sns.kdeplot(data=dados, x='Age', hue='Rotatividade', fill=True, common_norm=False, palette='Greys_r')
plt.xlabel('Idade do consumidor')
plt.ylabel('Densidade')
plt.title('Idade do consumidor - rotatividade versus nao rotatividade')
plt.tight_layout()
plt.show()

In [ ]:
# Compara a distribuicao de saldo do consumidor para as duas classes da variavel-alvo.
plt.figure(figsize=(15, 4))
sns.kdeplot(data=dados, x='Balance', hue='Rotatividade', fill=True, common_norm=False, palette='Greys_r')
plt.xlabel('Saldo do consumidor')
plt.ylabel('Densidade')
plt.title('Saldo do consumidor - rotatividade versus nao rotatividade')
plt.tight_layout()
plt.show()

In [ ]:
# Compara a distribuicao de escore de credito para as duas classes da variavel-alvo.
plt.figure(figsize=(15, 4))
sns.kdeplot(data=dados, x='EscoreCredito', hue='Rotatividade', fill=True, common_norm=False, palette='Greys_r')
plt.xlabel('Escore de credito')
plt.ylabel('Densidade')
plt.title('Escore de credito - rotatividade versus nao rotatividade')
plt.tight_layout()
plt.show()

In [ ]:
# Avalia a relacao entre o numero de produtos, o saldo e a rotatividade.
plt.figure(figsize=(16, 4))
ax = sns.barplot(x='NumDeProdutos', y='Balance', hue='Rotatividade', data=dados, palette='Greys_r', errorbar=None)
ax.set_title('Numero de produtos versus saldo')
ax.legend(title='Rotatividade', labels=['Nenhuma rotatividade', 'Rotatividade'])
plt.tight_layout()
plt.show()

## 6. Selecao de caracteristicas com Random Forest

A Random Forest cria varias arvores de decisao e combina suas previsoes. O atributo `feature_importances_` estima a contribuicao relativa de cada variavel para reduzir a impureza nas divisoes das arvores. Essa importancia e util como indicio exploratorio, mas nao deve ser interpretada automaticamente como causalidade.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

In [ ]:
# Para os algoritmos de Machine Learning, as categorias precisam ser convertidas em numeros.
# cat.codes atribui um codigo inteiro para cada categoria.
for coluna in ['Geography', 'Gender', 'HasCrCard', 'Rotatividade', 'MembroAtivo']:
    dados[coluna] = dados[coluna].astype('category').cat.codes

# Confirma os tipos apos a codificacao.
dados.dtypes

In [ ]:
# Define a coluna que o modelo deve prever.
alvo = 'Rotatividade'

# X contem as caracteristicas preditoras; y contem a resposta desejada.
X = dados.drop(alvo, axis=1)
y = dados[alvo]

# Divide os dados: 85% para treino e 15% para teste.
# stratify=y preserva aproximadamente a mesma proporcao das classes em ambos os conjuntos.
X_trein, X_teste, y_trein, y_teste = train_test_split(
    X, y, test_size=0.15, random_state=123, stratify=y
)

print('Treino:', X_trein.shape)
print('Teste :', X_teste.shape)

In [ ]:
# Cria o modelo com 500 arvores.
# random_state garante reprodutibilidade dos resultados.
floresta_aleatoria = RandomForestClassifier(n_estimators=500, random_state=1)

In [ ]:
# Ajusta o modelo aos dados de treinamento.
floresta_aleatoria.fit(X_trein, y_trein)

In [ ]:
# Recupera a importancia estimada de cada caracteristica.
importancia = floresta_aleatoria.feature_importances_
caracteristicas = X.columns

# argsort ordena os indices; [::-1] inverte para ordem decrescente.
indices = np.argsort(importancia)[::-1]

# Grafico das importancias ordenadas.
plt.figure(figsize=(15, 4))
plt.title('Importancia das caracteristicas usando Random Forest')
plt.bar(range(X_trein.shape[1]), importancia[indices], color='gray', align='center')
plt.xticks(range(X_trein.shape[1]), caracteristicas[indices], rotation='vertical', fontsize=12)
plt.xlim([-1, X_trein.shape[1]])
plt.tight_layout()
plt.show()

In [ ]:
# Monta uma tabela ordenada para interpretar a importancia das variaveis.
df_importancia_caracteristicas = pd.DataFrame({
    'Caracteristicas': caracteristicas,
    'Importancia': importancia
}).sort_values('Importancia', ascending=False).reset_index(drop=True)

df_importancia_caracteristicas

## 7. Classificacao com Regressao Logistica

A Regressao Logistica estima a probabilidade de cada cliente pertencer a classe `Rotatividade = 1`. O valor final de `score()` corresponde a acuracia no conjunto de teste. Em uma base desbalanceada, avalie tambem precision, recall, F1-score e matriz de confusao.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

In [ ]:
# Cria e ajusta o classificador logistico.
# max_iter foi aumentado para reduzir o risco de o otimizador nao convergir.
classificador = LogisticRegression(random_state=0, solver='lbfgs', max_iter=1000)
classificador.fit(X_trein, y_trein)

In [ ]:
# predict() retorna a classe prevista (0 ou 1) para cada amostra de teste.
previsoes = classificador.predict(X_teste)

# predict_proba() retorna as probabilidades estimadas para as classes 0 e 1.
probabilidades = classificador.predict_proba(X_teste)

print('Primeiras previsoes:', previsoes[:10])
print('Primeiras probabilidades [classe 0, classe 1]:')
print(probabilidades[:10])

In [ ]:
# Acuracia: proporcao total de previsoes corretas no conjunto de teste.
acuracia = classificador.score(X_teste, y_teste)
print(f'Acuracia no conjunto de teste: {acuracia:.3f}')

# Relatorio detalhado por classe.
print('\nRelatorio de classificacao:')
print(classification_report(y_teste, previsoes, target_names=['Sem rotatividade', 'Rotatividade']))

# Matriz de confusao: mostra acertos e erros por classe.
ConfusionMatrixDisplay.from_predictions(
    y_teste, previsoes, display_labels=['Sem rotatividade', 'Rotatividade'], cmap='Greys'
)
plt.title('Matriz de confusao - Regressao Logistica')
plt.show()

## Conclusao

O notebook apresenta um fluxo inicial de Ciencia de Dados para classificacao de rotatividade. Antes de usar o modelo em uma decisao real, recomenda-se: avaliar desbalanceamento das classes, padronizar variaveis numericas para a Regressao Logistica, testar validacao cruzada, comparar algoritmos e analisar metricas alem da acuracia.